In [ ]:
import numpy as np
import pandas as pd
from datetime import datetime

simulate the game

In [ ]:
np.random.seed(42) 
N = np.random.randint(30,41)
print(f"Number of players = {N}")

** Initialization **

When a player enters this game, he is assigned to an coupled vector, that is, (industry, historical yearly emissions over the past 10 years)
- power and utilities: very high emitters, historical emissions are around 850 - 1150 tCO2 per year
- heavy materials: high emitters, historical emissions are around 650 - 950 tCO2 per year
- manufacturing and chemicals: medium emitters, historical emissions are around 350 - 700 tCO2 per year
- transport: low emitters, historical emissions are around 150 - 450 per year

In [ ]:
industries = {
    "Power & Utilities": (850, 1150),
    "Heavy Materials": (650, 950),
    "Manufacturing & Chemicals": (350, 700),
    "Transport": (150, 450)
}

industry_names = list(industries.keys())

industry_probs = [0.25, 0.25, 0.25, 0.25]

Generate player data

In [ ]:
players = []

for pid in range(1, N + 1):
    # industry assignment
    industry = np.random.choice(industry_names, p=industry_probs)

    # past emission assignment
    low, high = industries[industry]

    latest = np. random.uniform(low, high)
    reduction = np.random.uniform(0.05, 0.2)
    oldest = latest * (1 + reduction)

    trend = np.linspace(oldest, latest, 10)
    noise = np.random.normal(0, 0.03 * latest, 10)
    emissions = np.round(trend + noise, 1)

    player = {
        "Player": f"P{pid}",
        "Industry": industry
    }

    for i in range(10):
        player[f"Year_{10-i}"] = emissions[i]

    players.append(player)

df = pd.DataFrame(players)

print(df.head())


In [ ]:
registration_time = datetime.now().strftime("%Y%m%d_%H%M%S")
df.to_excel(f"players_{registration_time}.xlsx", index=False)

** Cap mechanism **

We consider three modes of the cap mechanism. This mimic the development of ETS in EU and California, where the early model is grandfathering, then benchmarking, and nowadays auctioning. 
- Grandfathering: number of the assigned free credits are proportional to the previous emission history
- Benchmarking: an emission parameter is assigned to the industry, and the number of free credits is the production quantities times the emission parameter.
- Auctioning: no free credits. Purely auction-based. 

The implementation is aimed at Year 10 + 1. In general, one year has two stages. 
- Cap stage: the players decide on the credits that they want to buy
- Trade stage: the players trade among themselves

In [13]:
FREE_CREDIT_RATIO = 0.80
baseline_year = "Year_10"

total_baseline_emissions = df[baseline_year].sum()
free_credit_limit = total_baseline_emissions * FREE_CREDIT_RATIO

print("Total baseline emissions:", round(total_baseline_emissions, 1))
print("Free credit limit:", round(free_credit_limit, 1))

Total baseline emissions: 28332.7
Free credit limit: 22666.2


In [ ]:
#the moving time window
target_year = 11
history_window = 10
historical_years = list(range(target_year - history_window, target_year))

[1, 2, 3, 4, 5, 6, 7, 8, 9, 10]


In [ ]:
#Mode G: grandfathering 
years_cols = [f'Year_{year}' for year in historical_years]
df['past ten-year emissions'] = df[years_cols].sum(axis=1)  #note: this column is updating with the moving time window
total_past_ten_year_emissions = df['past ten-year emissions'].sum()

df[f"grandfathering free credits for Year_{target_year}"] = (
    df["past ten-year emissions"]
    / total_past_ten_year_emissions
    * free_credit_limit
).round(1)

In [ ]:
#Mode B: benchmarking

In [ ]:
#Mode A: auctioning

In [ ]:
#mode selection 
df[f'credits for Year_{target_year}'] = ...#select from three modes

** Realization of emissions at Year 11 **

The emission uncertainty is revealed after the cap stage, and before the trade. 


The emission is also industry-specific, as followed the initial setup.

In [22]:
def realize_target_year_emissions(df, target_year, industry_ranges):
    target_year_col = f"Year_{target_year}"
    df[target_year_col] = np.nan

    for industry, emission_range in industry_ranges.items():
        low, high = emission_range
        mask = df["Industry"] == industry

        df.loc[mask, target_year_col] = np.random.uniform(
            low=low,
            high=high,
            size=mask.sum()
        ).round(1)

    return df 

In [25]:
df= realize_target_year_emissions(df=df, target_year=target_year, industry_ranges = industries)

** Trade **

In [ ]:
df[f'Year_{target_year}'] - df[f'credits for Year_{target_year}'] ....

... Year 12 and onwards